# RQ3: automatic cross-domain comparison
Open this notebook in Colab and choose **Runtime > Run all**.
Setup automatically clones the `main` branch, installs dependencies, and prepares
the public dataset when valid CSVs are unavailable. No path edits, uploads,
Google Drive mount, or manual domain review are required.
GPU is preferred automatically; CPU is used when no GPU is available (slower).
The first dataset preparation downloads about 1.61 GB in addition to dependencies.
Compares coding-trained and math-trained classifiers on the SAME coding test set,
using CNN and RNN (four runs). Domains are keyword proxies from prompts, not human-reviewed labels.
The quick run uses at most 500 training, 100 validation and 200 test responses per family,
3 epochs and 128 tokens. This is an exploratory coding-versus-math RQ3 result.
Runtime depends on hardware, downloads, and available examples.
Results are saved under `/content/CMPSC-448-Midterm/rq3/experiments` in Colab.
Download the results before disconnecting; runtime resets erase local files.


In [ ]:
from pathlib import Path
import sys, subprocess, json, re
from datetime import datetime

REPO_URL = 'https://github.com/cucu071212/CMPSC-448-Midterm.git'
REPO_REF = 'main'
PROJECT_DIR = Path('/content/CMPSC-448-Midterm')
if not (PROJECT_DIR / 'train.py').is_file() and Path('train.py').is_file():
    PROJECT_DIR = Path.cwd()
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL,
                    str(PROJECT_DIR)], check=True)
PROJECT_DIR = PROJECT_DIR.resolve()
SOURCE_DIR = PROJECT_DIR / 'data'
RQ3_DIR = PROJECT_DIR / 'rq3'
for name in ('train.py', 'data.py', 'prepare_data.py', 'requirements.txt', 'src/cnn.py', 'src/rnn.py'):
    if not (PROJECT_DIR / name).is_file():
        raise RuntimeError(f'Code missing: {PROJECT_DIR / name}. Use the main branch containing the pipeline.')
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(PROJECT_DIR / 'requirements.txt')], check=True)


In [ ]:
import pandas as pd
import torch
from IPython.display import display, Image
from prepare_data import ensure_csvs, validate_csvs, COLUMNS, LABELS, SPLITS

SEED, EPOCHS, PATIENCE, BATCH_SIZE, MAX_LEN = 42, 3, 2, 64, 128
TARGET_DOMAIN, SOURCE_DOMAINS = 'coding', ['math']
SPLIT_CAPS = {'train': 500, 'validation': 100, 'test': 200}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (training will be slower)')
ensure_csvs(SOURCE_DIR, seed=SEED)
print('Original data:', validate_csvs(SOURCE_DIR))
frames = {split: pd.read_csv(SOURCE_DIR / f'{split}.csv', keep_default_na=False)
          for split in SPLITS}


## Automatic domain assignment
Use explicit coding and math keywords. Exclude prompts matching both or neither.
Save assigned domains and counts for reproducibility; no review upload is required.


In [ ]:
RULES = {
    'coding': re.compile(r'\b(python|javascript|typescript|programming|debug|sql|compiler|html|css)\b', re.I),
    'math': re.compile(r'\b(equation|integral|derivative|algebra|probability|calculus|theorem)\b', re.I),
}
def assign_domain(prompt):
    matches = [domain for domain, rule in RULES.items() if rule.search(prompt)]
    return matches[0] if len(matches) == 1 else 'excluded'
expected = pd.concat([frame[['prompt_id', 'LLM_Input']].assign(split=split)
                      for split, frame in frames.items()], ignore_index=True)
assignments = expected.drop_duplicates('prompt_id').copy()
assignments['domain'] = assignments.LLM_Input.map(assign_domain)
domain_map = assignments.set_index('prompt_id').domain
annotated = {split: frame.assign(domain=frame.prompt_id.map(domain_map))
             for split, frame in frames.items()}
counts = pd.concat([frame.assign(split=split) for split, frame in annotated.items()])
display(pd.crosstab([counts['split'], counts['domain']], counts.LLM_name))


## Matched experiment data
Preserve original splits, balance families, match fitting counts and reuse identical target test data.


In [ ]:
def balanced(frame, n, seed):
    return pd.concat([frame.loc[frame.LLM_name.eq(family)].sample(n=n, random_state=seed)
                      for family in LABELS], ignore_index=True).sample(
                          frac=1, random_state=seed).reset_index(drop=True)

def minimum_count(frame):
    return int(frame.LLM_name.value_counts().reindex(LABELS, fill_value=0).min())

def make_rq3_splits(annotated, target, sources, seed):
    if not sources or target in sources:
        raise ValueError('Source domains must be nonempty and exclude the target.')
    built = {'in_domain': {}, 'cross_domain': {}}
    for split in ('train', 'validation'):
        inside = annotated[split].loc[annotated[split].domain.eq(target)]
        outside = annotated[split].loc[annotated[split].domain.isin(sources)]
        n = min(minimum_count(inside), minimum_count(outside), globals().get('SPLIT_CAPS', {}).get(split, 10**9))
        if n < 1:
            raise ValueError(f'{split}: both conditions need keyword-assigned examples from all three families.')
        built['in_domain'][split] = balanced(inside, n, seed)
        built['cross_domain'][split] = balanced(outside, n, seed)
    target_test = annotated['test'].loc[annotated['test'].domain.eq(target)]
    n = min(minimum_count(target_test), globals().get('SPLIT_CAPS', {}).get('test', 10**9))
    if n < 1:
        raise ValueError('Target test needs keyword-assigned examples from all three families.')
    test = balanced(target_test, n, seed)
    for condition, parts in built.items():
        parts['test'] = test.copy()
        seen = set()
        for split in SPLITS:
            part = parts[split]
            ids = set(part.prompt_id)
            if seen & ids:
                raise ValueError(f'Prompt leakage in {condition}/{split}')
            if not ids <= set(annotated[split].prompt_id):
                raise ValueError('Original split membership changed.')
            seen.update(ids)
            if part.label.tolist() != part.LLM_name.map(LABELS).tolist():
                raise ValueError('Incorrect family labels.')
            if condition == 'cross_domain' and split != 'test' and part.domain.eq(target).any():
                raise ValueError('Target domain leaked into cross-domain fitting.')
    assert built['in_domain']['test'].equals(built['cross_domain']['test'])
    for split in ('train', 'validation'):
        assert built['in_domain'][split].LLM_name.value_counts().equals(
            built['cross_domain'][split].LLM_name.value_counts())
    return built

built = make_rq3_splits(annotated, TARGET_DOMAIN, SOURCE_DOMAINS, SEED)
SESSION_DIR = RQ3_DIR / 'experiments' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
SESSION_DIR.mkdir(parents=True, exist_ok=False)
DATA_DIRS = {}
manifest = {'target_domain': TARGET_DOMAIN, 'source_domains': SOURCE_DOMAINS,
            'seed': SEED, 'mode': 'output', 'domain_method': 'prompt_keyword_proxy',
            'split_caps_per_family': SPLIT_CAPS, 'epochs': EPOCHS, 'max_len': MAX_LEN,
            'conditions': {}}
for condition, parts in built.items():
    directory = SESSION_DIR / 'data' / condition
    directory.mkdir(parents=True)
    for split, part in parts.items():
        part[COLUMNS].to_csv(directory / f'{split}.csv', index=False)
    DATA_DIRS[condition] = directory
    manifest['conditions'][condition] = validate_csvs(directory)
assert (DATA_DIRS['in_domain'] / 'test.csv').read_bytes() == (DATA_DIRS['cross_domain'] / 'test.csv').read_bytes()
assignments.to_csv(SESSION_DIR / 'domain_assignments.csv', index=False)
counts.groupby(['split', 'domain', 'LLM_name']).size().to_csv(SESSION_DIR / 'domain_counts.csv')
(SESSION_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
OUTPUT_DIR = SESSION_DIR / 'results'
display(pd.DataFrame([dict(condition=c, split=s, **info)
                      for c, splits in manifest['conditions'].items() for s, info in splits.items()]))
print('Experiment folder:', SESSION_DIR)


In [ ]:
EXPERIMENTS = [('in_domain', 'cnn'), ('cross_domain', 'cnn'),
               ('in_domain', 'rnn'), ('cross_domain', 'rnn')]
RUNS = {}

def train_and_review(condition, model):
    RUNS.pop((condition, model), None)
    root = OUTPUT_DIR / condition
    before = set(root.glob(f'{model}_output/*/summary.csv'))
    command = [sys.executable, '-u', str(PROJECT_DIR / 'train.py'),
               '--model', model, '--mode', 'output', '--data-dir', str(DATA_DIRS[condition]),
               '--output-dir', str(root), '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
               '--batch-size', str(BATCH_SIZE), '--max-len', str(MAX_LEN),
               '--seed', str(SEED), '--device', DEVICE, '--num-workers', '0']
    with subprocess.Popen(command, cwd=str(PROJECT_DIR), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Training failed: {condition}/{model}; exit={returncode}')
    created = set(root.glob(f'{model}_output/*/summary.csv')) - before
    if len(created) != 1:
        raise RuntimeError('Could not identify exactly one new training result.')
    run_dir = next(iter(created)).parent
    RUNS[(condition, model)] = run_dir
    display(pd.read_csv(run_dir / 'summary.csv'))
    metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(metrics['classification_report']).T)
    display(Image(filename=str(run_dir / 'training_and_confusion.png')))
    display(pd.read_csv(run_dir / 'predictions.csv').head())
    print('Saved:', run_dir)
    return run_dir


In [ ]:
for condition, model in EXPERIMENTS:
    print(f'Running {condition} / {model}', flush=True)
    train_and_review(condition, model)


## RQ3 results
Positive F1 drop means fitting on math transfers worse to coding than fitting on coding.
Use the displayed sample counts and keyword-proxy domain definition in your report.


In [ ]:
tables, per_class, missing = [], [], []
for condition, model in EXPERIMENTS:
    run_dir = RUNS.get((condition, model))
    if run_dir is None:
        missing.append(f'{condition}/{model}')
        continue
    table = pd.read_csv(run_dir / 'summary.csv')
    table['condition'] = condition
    table['target_domain'] = TARGET_DOMAIN
    table['source_domains'] = ','.join(SOURCE_DOMAINS) if condition == 'cross_domain' else TARGET_DOMAIN
    table['run_dir'] = str(run_dir)
    tables.append(table)
    report = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))['classification_report']
    for family in LABELS:
        per_class.append(dict(model=model, condition=condition, family=family, **report[family]))
if tables:
    comparison = pd.concat(tables, ignore_index=True)
    comparison.to_csv(SESSION_DIR / 'comparison.csv', index=False)
    display(comparison)
    class_table = pd.DataFrame(per_class)
    class_table.to_csv(SESSION_DIR / 'per_family.csv', index=False)
    display(class_table)
    drops = comparison.pivot(index='model', columns='condition', values='test_macro_f1')
    if {'in_domain', 'cross_domain'} <= set(drops.columns):
        drops['f1_drop'] = drops['in_domain'] - drops['cross_domain']
        drops.to_csv(SESSION_DIR / 'generalization_gap.csv')
        display(drops)
if missing:
    print('Incomplete comparison; missing:', ', '.join(missing))
else:
    print('All four experiments are included.')

print('Results saved:', SESSION_DIR)
